In [4]:
import requests
import pandas as pd

BASE_URL = "https://gamma-api.polymarket.com/markets"

def fetch_markets(limit=500):
    params = {
        "limit": limit,
        "offset": 0,
        "closed": "true"
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()

data = fetch_markets()
print(data[0].keys())
keys = list(data[0].keys())
for key in keys:
    print(key, data[0].get(key))
print(len(data))

dict_keys(['id', 'question', 'conditionId', 'slug', 'twitterCardImage', 'endDate', 'category', 'liquidity', 'image', 'icon', 'description', 'outcomes', 'outcomePrices', 'volume', 'active', 'marketType', 'closed', 'marketMakerAddress', 'updatedBy', 'createdAt', 'updatedAt', 'closedTime', 'mailchimpTag', 'archived', 'restricted', 'volumeNum', 'liquidityNum', 'endDateIso', 'hasReviewedDates', 'readyForCron', 'volume24hr', 'volume1wk', 'volume1mo', 'volume1yr', 'clobTokenIds', 'fpmmLive', 'volume1wkAmm', 'volume1moAmm', 'volume1yrAmm', 'volume1wkClob', 'volume1moClob', 'volume1yrClob', 'events', 'creator', 'ready', 'funded', 'cyom', 'competitive', 'pagerDutyNotificationEnabled', 'approved', 'rewardsMinSize', 'rewardsMaxSpread', 'spread', 'oneDayPriceChange', 'oneHourPriceChange', 'oneWeekPriceChange', 'oneMonthPriceChange', 'oneYearPriceChange', 'lastTradePrice', 'bestBid', 'bestAsk', 'clearBookOnStart', 'manualActivation', 'negRiskOther', 'umaResolutionStatuses', 'pendingDeployment', 'dep

In [5]:
import json

def is_binary_market(market):
    outcomes = market.get("outcomes", [])
    outcomes = json.loads(outcomes)
    return len(outcomes) == 2

def is_resolved(market):
    return market.get("closed") is True
    # outcomePricesStr = json.loads(market.get("outcomePrices"))
    # outcomePrices = [float(x) for x in outcomePricesStr]
    # return (outcomePrices == [0.0, 1.0] or outcomePrices == [1.0, 0.0])

def is_valid_market(market):
    return (
        is_binary_market(market) and
        is_resolved(market) and
        (market.get("volumeNum", 0) > 0)
    )

filtered = [m for m in data if is_valid_market(m)]

print("Filtered markets:", len(filtered))

Filtered markets: 471


In [6]:
BASE_URL = "https://data-api.polymarket.com"

def get_recent_trades(limit=20):
    url = f"{BASE_URL}/trades"
    params = {"limit": limit}

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def main():
    trades = get_recent_trades(limit=20)

    df = pd.DataFrame(trades)
    print("Columns:", df.columns.tolist())
    print()
    print(df.head())

if __name__ == "__main__":
    main()

Columns: ['proxyWallet', 'side', 'asset', 'conditionId', 'size', 'price', 'timestamp', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'outcomeIndex', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized', 'transactionHash']

                                  proxyWallet  side  \
0  0xd044bc25e2ece78affecf1aa91520a31d6fb2f75  SELL   
1  0x84cfffc3f16dcc353094de30d4a45226eccd2f63   BUY   
2  0xc41b4d12d78b374b55e4562b4f3b5c9b10d77e45   BUY   
3  0x0c46eb813826a4106f77b3aaf608943228423191  SELL   
4  0x12fcc5dd1cfe556eaee7c592c1f488b78d7f3a22   BUY   

                                               asset  \
0  5179715774304650421854161668175159784546805590...   
1  8962887998343866669322983609500790184439717269...   
2  3487737759656637510052401454254809955253027146...   
3  1270012870270253366780745394100419118367812601...   
4  3563264031556944556584453710841330444389361735...   

                                         conditionId       size  price  \
0  0x0b4cc3b739e1

In [7]:
BASE_URL = "https://data-api.polymarket.com"

def fetch_price_history(token_id, interval="1h", fidelity=1):
    """
    interval: '1m', '5m', '1h', '1d'
    fidelity: controls granularity (keep = 1 for now)
    """
    url = f"{BASE_URL}/trades"
    
    params = {
        "token": token_id,
        "interval": interval,
        "fidelity": fidelity
    }

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data)
    
    # if not df.empty:
    #    df["t"] = pd.to_datetime(df["t"], unit="s")
    #    df = df.rename(columns={"t": "timestamp", "p": "price"})
    
    return df

In [8]:
# tokens = filtered[0].get("clobTokenIds").translate(str.maketrans('', '', '[]",')).split()
tokens = json.loads(filtered[0].get("clobTokenIds", "[]"))
print(tokens[0])
df = fetch_price_history(tokens[0])
print(df['price'].head())

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

df = df.sort_values("timestamp")

price_series = (
    df.set_index("timestamp")["price"]
    .resample("1h")
    .last()
    .ffill()
)

53135072462907880191400140706440867753044989936304433583131786753949599718775
0    0.990
1    0.250
2    0.410
3    0.999
4    0.999
Name: price, dtype: float64


In [9]:
def get_price_at_horizon(price_df, resolution_time, days_before):
    target_time = resolution_time - pd.Timedelta(days=days_before)

    df_before = price_df[price_df["timestamp"] <= target_time]

    if df_before.empty:
        return None

    return df_before.iloc[-1]["price"]

def build_market_features(price_df, resolution_time):
    return {
        "p_30d": get_price_at_horizon(price_df, resolution_time, 30),
        "p_7d": get_price_at_horizon(price_df, resolution_time, 7),
        "p_1d": get_price_at_horizon(price_df, resolution_time, 1),
    }

def get_outcome(market):
    return int(market["outcome"] == "Yes")  # adjust if needed

In [ ]:
dataset = []

for market in filtered:
    try:
        tokens = json.loads(market["clobTokenIds"])
        token_id = tokens[0]

        price_df = fetch_price_history(token_id)
        if price_df is None:
            print("no price history")
            continue

        print("price history")
        resolution_time = pd.to_datetime(market["endDate"])
        outcome = get_outcome(market)

        features = build_market_features(price_df, resolution_time)

        row = {
            "market_id": market["id"],
            "outcome": outcome,
            **features
        }

        dataset.append(row)

    except Exception as e:
        continue

df_final = pd.DataFrame(dataset)
print(df_final.head())